In [1]:
"""
MVP Fact-checking vietnamita — Ultra ligero, sin GPU
Peso final: ~130-150 MB | Sin GPU requerida
pip install torch transformers scikit-learn kagglehub
"""

import json, random
import numpy as np
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ── CONFIG ───────────────────────────────────────────────────────────────────
CONFIG = {
    "model_name": "distilbert-base-multilingual-cased",
    "max_length": 96,
    "batch_size": 16,
    "epochs": 3,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "seed": 42,
    "output_dir": "./outputs/mvp-fact-checking",
    "evidence_words": 60,
    "max_samples": 2000,
}

LABEL2ID = {"SUPPORTED": 0, "REFUTED": 1}
ID2LABEL = {0: "SUPPORTED", 1: "REFUTED"}

# ── SEMILLA ──────────────────────────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

# ── CARGA DEL DATASET ────────────────────────────────────────────────────────
def load_dataset_from_dir(dataset_dir: str):
    dataset_dir = Path(dataset_dir)
    samples = []

    all_files = list(dataset_dir.rglob("*.json"))
    files     = [f for f in all_files if "HIGH_CONFIDENCE" in f.name] or all_files
    print(f"Archivos: {[f.name for f in files]}")

    for fpath in files:
        try:
            content = json.loads(fpath.read_text(encoding="utf-8"))
            for item in (content if isinstance(content, list) else [content]):
                label    = item.get("label", "").upper().strip()
                if label not in LABEL2ID:
                    continue
                claim    = item.get("claim", "")
                evidence = item.get("contexts", item.get("evidence", ""))
                if isinstance(evidence, list):
                    evidence = " ".join(e if isinstance(e, str) else " ".join(e) for e in evidence)
                if claim and evidence:
                    samples.append({"claim": claim.strip(), "evidence": evidence.strip(), "label": label})
        except Exception as e:
            print(f"  Error {fpath.name}: {e}")

    random.shuffle(samples)
    if CONFIG["max_samples"]:
        samples = samples[:CONFIG["max_samples"]]

    counts = {}
    for s in samples:
        counts[s["label"]] = counts.get(s["label"], 0) + 1
    print(f"Total: {len(samples)} | {counts}")
    return samples

def balancear_clases(samples):
    supported = [s for s in samples if s["label"] == "SUPPORTED"]
    refuted = [s for s in samples if s["label"] == "REFUTED"]

    n = min(len(supported), len(refuted))

    balanced = supported[:n] + refuted[:n]
    random.shuffle(balanced)

    print(f"Dataset balanceado: {len(balanced)} muestras")
    print(f"SUPPORTED: {n} | REFUTED: {n}")

    return balanced

def split_dataset(samples):
    n = len(samples)
    return samples[:int(n*0.8)], samples[int(n*0.8):int(n*0.9)], samples[int(n*0.9):]

# ── DATASET ──────────────────────────────────────────────────────────────────
class FactCheckDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.samples   = samples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item     = self.samples[idx]
        evidence = " ".join(item["evidence"].split()[:CONFIG["evidence_words"]])
        enc      = self.tokenizer(
            item["claim"], evidence,
            max_length=CONFIG["max_length"],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(LABEL2ID[item["label"]], dtype=torch.long),
        }

# ── TRAIN / EVAL ─────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, preds_all, labels_all = 0, [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for batch in loader:
            out  = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
            if training:
                optimizer.zero_grad()
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += out.loss.item()
            preds_all.extend(out.logits.argmax(-1).numpy())
            labels_all.extend(batch["label"].numpy())

    return total_loss / len(loader), accuracy_score(labels_all, preds_all), f1_score(labels_all, preds_all, average="macro"), preds_all, labels_all

# ── INFERENCIA ───────────────────────────────────────────────────────────────
def predict(claim: str, evidence: str, model, tokenizer) -> dict:
    """Output formato torneo: {'predicted_label': 'SUPPORTED' | 'REFUTED'}"""
    model.eval()
    enc = tokenizer(claim, evidence, max_length=CONFIG["max_length"], padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        pred = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits.argmax(-1).item()
    return {"predicted_label": ID2LABEL[pred]}

# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    set_seed(CONFIG["seed"])
    print("Device: CPU")

    import kagglehub
    dataset_dir = kagglehub.dataset_download("haisemei/fact-checking-dataset-label")

    samples = load_dataset_from_dir(dataset_dir)
    if not samples:
        raise ValueError("No se cargaron muestras.")

    samples = balancear_clases(samples)

    train_data, val_data, test_data = split_dataset(samples)
    print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

    print(f"\nCargando {CONFIG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    model     = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"], num_labels=2, id2label=ID2LABEL, label2id=LABEL2ID
    )

    train_loader = DataLoader(FactCheckDataset(train_data, tokenizer), batch_size=CONFIG["batch_size"], shuffle=True)
    val_loader   = DataLoader(FactCheckDataset(val_data,   tokenizer), batch_size=CONFIG["batch_size"])
    test_loader  = DataLoader(FactCheckDataset(test_data,  tokenizer), batch_size=CONFIG["batch_size"])

    optimizer   = AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
    total_steps = len(train_loader) * CONFIG["epochs"]
    scheduler   = get_linear_schedule_with_warmup(optimizer, int(total_steps * CONFIG["warmup_ratio"]), total_steps)

    output_dir   = Path(CONFIG["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    best_val_acc = 0

    print("\n" + "="*50)
    for epoch in range(1, CONFIG["epochs"] + 1):
        tr_loss, tr_acc, _,      _, _ = run_epoch(model, train_loader, optimizer, scheduler)
        vl_loss, vl_acc, vl_f1,  _, _ = run_epoch(model, val_loader)
        print(f"Época {epoch}/{CONFIG['epochs']} | Train Loss {tr_loss:.4f} Acc {tr_acc:.4f} | Val Loss {vl_loss:.4f} Acc {vl_acc:.4f} F1 {vl_f1:.4f}")

        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            model.save_pretrained(output_dir / "best_model")
            tokenizer.save_pretrained(output_dir / "best_model")
            print(f"  ✓ Guardado (val_acc={vl_acc:.4f})")

    # ── Evaluación final ─────────────────────────────────────────────────────
    print("\n" + "="*50)
    _, test_acc, test_f1, test_preds, test_labels = run_epoch(model, test_loader)
    print(f"Test Accuracy: {test_acc:.4f} | F1: {test_f1:.4f}")
    print(classification_report(test_labels, test_preds, target_names=["SUPPORTED", "REFUTED"]))

    # ── Cuantización → reduce ~4x el peso ────────────────────────────────────
    print("\nCuantizando modelo...")
    best_model = AutoModelForSequenceClassification.from_pretrained(output_dir / "best_model")
    best_model.eval()
    quantized  = torch.quantization.quantize_dynamic(best_model, {torch.nn.Linear}, dtype=torch.qint8)

    quant_path = output_dir / "model_quantized"
    quant_path.mkdir(parents=True, exist_ok=True)
    torch.save(quantized.state_dict(), quant_path / "model.pt")
    tokenizer.save_pretrained(quant_path)

    # ── Tamaños ───────────────────────────────────────────────────────────────
    size_original  = sum(f.stat().st_size for f in (output_dir / "best_model").rglob("*") if f.is_file())
    size_quantized = sum(f.stat().st_size for f in quant_path.rglob("*") if f.is_file())
    print(f"Original:   {size_original  / 1e6:.1f} MB")
    print(f"Cuantizado: {size_quantized / 1e6:.1f} MB")

    # ── Ejemplo de inferencia con modelo cuantizado ───────────────────────────
    print("\n" + "="*50)
    res = predict("Hà Nội là thủ đô của Việt Nam.", "Hà Nội là thủ đô và là thành phố lớn nhất của Việt Nam.", quantized, tokenizer)
    print(f"Ejemplo inferencia: {res}")
    print(f"\nModelo listo en: {quant_path}")

if __name__ == "__main__":
    main()

Device: CPU


100%|██████████| 41.3M/41.3M [00:00<00:00, 130MB/s]

Extracting files...


Archivos: ['HIGH_CONFIDENCE_train_dataset.json', 'HIGH_CONFIDENCE_test_dataset.json', 'HIGH_CONFIDENCE_validation_dataset.json', 'HIGH_CONFIDENCE_converted_dataset.json']
Total: 2000 | {'REFUTED': 693, 'SUPPORTED': 1307}
Dataset balanceado: 1386 muestras
SUPPORTED: 693 | REFUTED: 693
Train: 1108 | Val: 139 | Test: 139

Cargando distilbert-base-multilingual-cased...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Guardado (val_acc=0.6906)
Época 3/3 | Train Loss 0.5729 Acc 0.7229 | Val Loss 0.5850 Acc 0.6906 F1 0.6890

Test Accuracy: 0.6043 | F1: 0.6042
              precision    recall  f1-score   support

   SUPPORTED       0.56      0.67      0.61        64
     REFUTED       0.66      0.55      0.60        75

    accuracy                           0.60       139
   macro avg       0.61      0.61      0.60       139
weighted avg       0.61      0.60      0.60       139


Cuantizando modelo...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/tmp/ipykernel_9880/3868606901.py:206: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized  = torch.quantization.quantize_dynamic(best_model, {torch.nn.Linear}, dtype=torch.qint8)


Original:   544.2 MB
Cuantizado: 415.1 MB

Ejemplo inferencia: {'predicted_label': 'SUPPORTED'}

Modelo listo en: outputs/mvp-fact-checking/model_quantized


In [2]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    "distilbert-base-multilingual-cased"
)

config.save_pretrained(
    "outputs/mvp-fact-checking/model_quantized"
)

print("config.json guardado")

config.json guardado


In [6]:
import shutil

shutil.make_archive(
    "model_quantized",
    "zip",
    "outputs/mvp-fact-checking/model_quantized"
)

'/content/model_quantized.zip'

In [12]:
import kagglehub

dataset_dir = kagglehub.dataset_download("haisemei/fact-checking-dataset-label")

samples = load_dataset_from_dir(dataset_dir)

Using Colab cache for faster access to the 'fact-checking-dataset-label' dataset.
Archivos: ['HIGH_CONFIDENCE_test_dataset.json', 'HIGH_CONFIDENCE_converted_dataset.json', 'HIGH_CONFIDENCE_validation_dataset.json', 'HIGH_CONFIDENCE_train_dataset.json']
Total: 2000 | {'SUPPORTED': 1300, 'REFUTED': 700}


In [13]:
refuted = [s for s in samples if s["label"] == "REFUTED"]

for i, s in enumerate(refuted[:5], 1):
    evidence_corto = " ".join(s["evidence"].split()[:60])

    print(f"\n--- REFUTED {i} ---")
    print(f"""{{
  "claim": "{s["claim"]}",
  "context": "{evidence_corto}"
}}""")


--- REFUTED 1 ---
{
  "claim": "Vào tháng 10 năm 2023, hồ thủy điện Dầu Tiếng đã tiến hành xả tràn với lưu lượng 500 m3/s để giảm áp lực và cắt lũ hạ du.",
  "context": "Theo thông báo mới nhất, đợt xả tràn này bắt đầu từ 7h ngày 25/11 đến 7h ngày 2/12. Lưu lượng xả được điều chỉnh linh hoạt từ 36 m3/s đến 200 m3/s tùy theo diễn biến thực tế. Sau thời gian này, hồ sẽ duy trì dòng chảy sau đập ở mức tối thiểu 36 m3/s theo quy trình"
}

--- REFUTED 2 ---
{
  "claim": "Người phát ngôn Bộ Ngoại giao Trung Quốc Quách Gia Côn tuyên bố vào ngày 24/4 rằng Trung Quốc và Mỹ chưa có bất kỳ cuộc tham vấn hay đàm phán nào về vấn đề thuế quan.",
  "context": "Bộ Thương mại Trung Quốc vừa tuyên bố với Mỹ: Nếu đàm phán, cánh cửa luôn rộng mở; còn nếu đối đầu, Trung Quốc sẽ sẵn sàng đáp trả đến cùng. Chiều 10-4, tại buổi họp báo thường kỳ, người phát ngôn Bộ Thương mại Trung Quốc Hà Vịnh Tiền cho biết thời gian gần đây Mỹ đã nhiều lần"
}

--- REFUTED 3 ---
{
  "claim": "Theo Cục Cảnh sát giao thông, v